In [9]:
import gradio as gr
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from PIL import Image

In [10]:
# ----------------------------
# Object, first time run
# ----------------------------

@dataclass(frozen=True)
class Obj:
    name: str
    color: str
    shape: str  # "cube", "pyramid", "block" (generic)
    size: str  # "small", "medium", "large"


In [11]:
# ----------------------------
# World, first time run
# ----------------------------

@dataclass
class World:
    objects: Dict[str, Obj]
    objects_inside_box: List[str] # list of object names that are inside the box
    whoIsUnder: Dict[str, Optional[str]] # on[x] = y means x is on y; None means on table
    #If you have on["block_a"] = "block_b", it means block_a is physically sitting on top of block_b
    holding: Optional[str] = None

    def is_clear(self, obj_name: str) -> bool:
        """True if nothing is on top of obj_name."""
        # return all(support != obj_name for support in self.on.values())
        for support in self.whoIsUnder.values():
            if support == obj_name:
                return False
        return True

    def top_of(self, obj_name: str) -> Optional[str]:
        """Return object that is on top of obj_name (if any)."""
        for x, support in self.whoIsUnder.items():
            print(x,support)
            if support == obj_name:
                return x
        return None

    def describe(self) -> str:
        #describes all relationships between objects and if world agent is holding something
        lines = []
        for x in sorted(self.objects):
            support = self.whoIsUnder.get(x, None)
            if support is None:
                lines.append(f"{x} is on the table")
            else:
                lines.append(f"{x} is on {support}")
        lines.append(f"Holding: {self.holding}")
        return "\n".join(lines)

    ##########
    
    def pickup(self, x: str) -> None:
        #pick up obj if already not holding anything and if object isnt under something else
        if self.holding is not None:
            raise RuntimeError("Already holding something.")
        if not self.is_clear(x):
            raise RuntimeError(f"Cannot pick up {x}: not clear.")
        self.holding = x
        self.whoIsUnder[x] = None

    def put_on(self, x: str, y: str) -> None:
        #put down obj on another obj, if holding something and if target object isnt under something else / blocked
        if self.holding != x:
            raise RuntimeError(f"Not holding {x}.")
        if not self.is_clear(y):
            raise RuntimeError(f"Cannot place on {y}: {y} not clear.")
        self.whoIsUnder[x] = y
        self.holding = None
    
    def put_down(self, x: str) -> None:
        #put down obj on table, if holding something and if target object isnt under something else / blocked
        if self.holding != x:
            raise RuntimeError(f"Not holding {x}.")
        self.whoIsUnder[x] = None
        self.holding = None


# Initialize world state
def create_initial_world():
    return World(
        objects={
            "c1": Obj("c1", "red", "cube", "medium"),
            "b2": Obj("b2", "yellow", "cube", "medium"),
            "p1": Obj("p1", "blue", "pyramid", "large"),
            "c2": Obj("c2", "green", "pyramid", "small"),
            "l1": Obj("l1", "cyan", "circle", "medium"),
            "r1": Obj("r1", "magenta", "rectangle", "large"),
            "s1": Obj("s1", "yellow", "pentagon", "medium"),
            "s2": Obj("s2", "red", "pentagon", "small"),
            "l2": Obj("l2", "blue", "circle", "small"),
            "l3": Obj("l3", "green", "circle", "large"),
            "b3": Obj("b3", "cyan", "cube", "small"),
            "c3": Obj("c3", "magenta", "pyramid", "medium"),
            "c4": Obj("c4", "yellow", "cube", "small"),
        },
        whoIsUnder={
            "p1": "c1",
            "c1": None,
            "b2": None,
            "c2": None,
            "l1": "c2",
            "r1": None,
            "s1": None,
            "s2": None,
            "l2": None,
            "l3": None,
            "b3": None,
            "c3": None,
            "c4": None
        },
        objects_inside_box=["c1", "p1"]
    )

# Global world state
world_state = create_initial_world()

In [12]:
# ----------------------------------------
# STEP 3) Planning
# ----------------------------------------

def plan_pickup(world: World, x: str) -> List[Tuple[str, str, Optional[str]]]:
    """Plan to pick up x. If x is not clear, move blockers to table first."""
    plan = []
    blocker = world.top_of(x)
    if blocker is not None:
        plan += plan_pickup(world, blocker)
        plan.append(("put_on", blocker, None))
    plan.append(("pickup", x, None))
    return plan


def plan_put(world: World, x: str, y: str) -> List[Tuple[str, str, Optional[str]]]:
    """Plan to put x on y, ensuring both are feasible."""
    plan = []
    blocker = world.top_of(y)
    if blocker is not None:
        plan += plan_pickup(world, blocker)
        plan.append(("put_on", blocker, None))
    plan += plan_pickup(world, x)
    plan.append(("put_on", x, y))
    return plan


def execute_plan(world: World, plan: List[Tuple[str, str, Optional[str]]]) -> None:
    for action, obj, target in plan:
        if action == "pickup":
            world.pickup(obj)
        elif action == "put_on":
            if target is None:
                if world.holding != obj:
                    raise RuntimeError("Planner/executor mismatch.")
                world.whoIsUnder[obj] = None
                world.holding = None
            else:
                world.put_on(obj, target)
        elif action == "put_down":
            if world.holding != obj:
                raise RuntimeError("Planner/executor mismatch.")
            world.whoIsUnder[obj] = None
            world.holding = None
        else:
            raise ValueError(f"Unknown action: {action}")

In [13]:
# ----------------------------------------
# STEP 2) Parsing
# ----------------------------------------

COLORS = {"red", "green", "blue", "yellow", "magenta", "cyan"}
SHAPES = {"pyramid", "cube", "rectangle", "circle", "pentagon", "block"}
SIZES = {"small", "medium", "large"}

def parse_command(text: str) -> dict:
    """Minimal rule-based parser for demo commands."""
    t = text.lower().replace("?", "").strip()
    tokens = t.split()

    if t.startswith("pick up"):
        color = next((w for w in tokens if w in COLORS), None)
        shape = next((w for w in tokens if w in SHAPES), "block")
        size = next((w for w in tokens if w in SIZES), "medium")
        return {"intent": "PICKUP", "ref": {"color": color, "shape": shape, "size": size}}

    if t.startswith("put"):
        if "on" in tokens:
            on_i = tokens.index("on")
            left = tokens[1:on_i]
            right = tokens[on_i+1:]

            color_x = next((w for w in left if w in COLORS), None)
            shape_x = next((w for w in left if w in SHAPES), "block")
            size_x = next((w for w in left if w in SIZES), "medium")
            color_y = next((w for w in right if w in COLORS), None)
            shape_y = next((w for w in right if w in SHAPES), "block")
            size_y = next((w for w in right if w in SIZES), "medium")

            return {
            "intent": "PUT_ON",
            "x": {"color": color_x, "shape": shape_x, "size": size_x},
            "y": {"color": color_y, "shape": shape_y, "size": size_y},
        }
        elif "down" in tokens:
            #put down
            on_i = tokens.index("down")
            left = tokens[1:on_i]
            right = tokens[on_i+1:]

            color_x = next((w for w in left if w in COLORS), None)
            shape_x = next((w for w in left if w in SHAPES), "block")
            size_x = next((w for w in left if w in SIZES), "medium")
            color_y = next((w for w in right if w in COLORS), None)
            shape_y = next((w for w in right if w in SHAPES), "block")
            size_y = next((w for w in right if w in SIZES), "medium")
            
            return {
            "intent": "PUT_DOWN",
            "x": {"color": color_x, "shape": shape_x, "size": size_x},
            "y": {"color": color_y, "shape": shape_y, "size": size_y},
        }

    raise ValueError("Unknown command format.")

#finds object names matching the requested properties, and returns a list of matches
def resolve_ref(world: World, color: Optional[str], shape: Optional[str]) -> List[str]:
    """Return object names matching the requested properties."""
    matches = []
    for name, obj in world.objects.items():
        if color is not None and obj.color != color:
            continue
        if shape is not None:
            if shape != "block" and obj.shape != shape:
                continue
        matches.append(name)
    return matches

#returns single, no or multiple matches for the given description of an object
def choose_unique(matches: List[str], what: str) -> str:
    if not matches:
        raise ValueError(f"I can't find any {what} that matches your description.")
    if len(matches) > 1:
        raise ValueError(f"I don't understand which {what} you mean: {matches}")
    return matches[0]



#this calls everything
def interpret_and_act(world: World, utterance: str) -> str:
    """Execute command and return response message."""
    parsed = parse_command(utterance)

    if parsed["intent"] == "PICKUP":
        ref = parsed["ref"]
        matches = resolve_ref(world, ref["color"], ref["shape"])
        x = choose_unique(matches, f"{ref['color'] or ''} {ref['shape']}".strip())
        plan = plan_pickup(world, x)
        execute_plan(world, plan)
        return f"OK. Picked up {x}. Plan: {plan}"

    elif parsed["intent"] == "PUT_ON":
        mx = resolve_ref(world, parsed["x"]["color"], parsed["x"]["shape"])
        my = resolve_ref(world, parsed["y"]["color"], parsed["y"]["shape"])
        x = choose_unique(mx, f"{parsed['x']['color'] or ''} {parsed['x']['shape']}".strip())
        y = choose_unique(my, f"{parsed['y']['color'] or ''} {parsed['y']['shape']}".strip())
        plan = plan_put(world, x, y)
        execute_plan(world, plan)
        return f"OK. Put {x} on {y}. Plan: {plan}"
    
    elif parsed["intent"] == "PUT_DOWN":
        mx = resolve_ref(world, parsed["x"]["color"], parsed["x"]["shape"])
        x = choose_unique(mx, f"{parsed['x']['color'] or ''} {parsed['x']['shape']}".strip())
        plan = [("put_down", x, None)]
        execute_plan(world, plan)
        return f"OK. Put down {x}. Plan: {plan}"

    else:
        raise ValueError("Unsupported intent.")


In [14]:
# ----------------------------------------
# STEP 4) Visualization
# ----------------------------------------

size_map = {
    "small": 0.3,
    "medium": 0.5,
    "large": 0.8
}

#helper
def get_width(obj):
    s = size_map[obj.size]
    
    if obj.shape == "rectangle":
        return s
    elif obj.shape == "circle":
        return s
    elif obj.shape == "pyramid":
        return s
    elif obj.shape == "pentagon":
        return s
    else:  # cube/block
        return s

def get_height(obj):
    s = size_map[obj.size]
    
    if obj.shape == "pyramid":
        return s * 1.2
    elif obj.shape == "cube":
        return s
    elif obj.shape == "rectangle":
        return s / 1.5
    elif obj.shape == "circle":
        return s
    elif obj.shape == "pentagon":
        return s * 0.6
    else:  # block
        return s


def visualize_world(world: World) -> Image.Image:
    """Create a visual representation of the blocks world."""
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_xlim(-1, 10)
    ax.set_ylim(-0.5, 5)
    ax.set_aspect('equal')
    ax.axis('off')
    
    # Build stacks for each object on the table
    stacks = {}
    x_pos = 0
    
    # Find all objects on the table
    on_table = [name for name, support in world.whoIsUnder.items() if support is None and name != world.holding]
    
    for table_obj in sorted(on_table):
        stack = [table_obj]
        # Build stack upward
        current = table_obj
        while True:
            top = world.top_of(current)
            if top is None:
                break
            stack.append(top)
            current = top
        stacks[table_obj] = stack
        
    # Draw each stack
    x_pos = 0.5
    for base_obj, stack in stacks.items():
        y_pos = 0
        center_x = x_pos
        for obj_name in stack:
            obj = world.objects[obj_name]
            
            if obj.shape == "pyramid":
                s = size_map[obj.size]
                width = get_width(obj)
                left = center_x - width / 2

                points = [
                    [left, y_pos],                 # bottom left
                    [left + width, y_pos],         # bottom right
                    [center_x, y_pos + s * 1.2]    # top (centered)
                ]

                triangle = patches.Polygon(
                    points,
                    closed=True,
                    edgecolor='black',
                    facecolor=obj.color,
                    linewidth=2
                )
                ax.add_patch(triangle)

                # center text properly
                ax.text(
                    center_x,
                    y_pos + (s * 1.2) / 2,
                    obj_name,
                    ha='center',
                    va='center',
                    fontsize=10,
                    fontweight='bold'
                )

                y_pos += get_height(obj)
            
            elif obj.shape == "cube":
                s = size_map[obj.size]
                left = center_x - s / 2

                rect = patches.Rectangle(
                    (left, y_pos),
                    s,
                    s,
                    edgecolor='black',
                    facecolor=obj.color,
                    linewidth=2
                )
                ax.add_patch(rect)

                ax.text(
                    center_x,
                    y_pos + s / 2,
                    obj_name,
                    ha='center',
                    va='center',
                    fontsize=10,
                    fontweight='bold'
                )

                y_pos += get_height(obj)
            
            elif obj.shape == "rectangle":
                s = size_map[obj.size]
                w = s
                h = s / 1.5

                left = center_x - w / 2

                rect = patches.Rectangle(
                    (left, y_pos),
                    w,
                    h,
                    edgecolor='black',
                    facecolor=obj.color,
                    linewidth=2
                )
                ax.add_patch(rect)

                ax.text(
                    center_x,
                    y_pos + h / 2,
                    obj_name,
                    ha='center',
                    va='center',
                    fontsize=10,
                    fontweight='bold'
                )

                y_pos += get_height(obj)
            
            elif obj.shape == "circle":
                s = size_map[obj.size]
                r = s / 2

                circle = patches.Circle(
                    (center_x, y_pos + r),
                    r,
                    edgecolor='black',
                    facecolor=obj.color,
                    linewidth=2
                )
                ax.add_patch(circle)

                ax.text(
                    center_x,
                    y_pos + r,
                    obj_name,
                    ha='center',
                    va='center',
                    fontsize=10,
                    fontweight='bold'
                )

                y_pos += get_height(obj)
            
            elif obj.shape == "pentagon":
                s = size_map[obj.size]
                left = center_x - s / 2

                points = [
                    [center_x, y_pos + s * 0.6],          # top
                    [left + s * 0.9, y_pos + s * 0.4],    # right upper
                    [left + s * 0.7, y_pos],              # right lower
                    [left + s * 0.3, y_pos],              # left lower
                    [left + s * 0.1, y_pos + s * 0.4],    # left upper
                ]

                pentagon = patches.Polygon(
                    points,
                    closed=True,
                    edgecolor='black',
                    facecolor=obj.color,
                    linewidth=2
                )
                ax.add_patch(pentagon)

                ax.text(
                    center_x,
                    y_pos + s / 2,
                    obj_name,
                    ha='center',
                    va='center',
                    fontsize=10,
                    fontweight='bold'
                )

                y_pos += get_height(obj)
            
            else:  #block
                s = 0.5
                left = center_x - s / 2

                rect = patches.Rectangle(
                    (left, y_pos),
                    s,
                    s,
                    edgecolor='black',
                    facecolor=obj.color,
                    linewidth=2
                )
                ax.add_patch(rect)

                ax.text(
                    center_x,
                    y_pos + s / 2,
                    obj_name,
                    ha='center',
                    va='center',
                    fontsize=10,
                    fontweight='bold'
                )

                y_pos += get_height(obj)
        
        max_width = max(get_width(world.objects[name]) for name in stack)
        x_pos += max_width + 0.5
    
    # Draw held object
    if world.holding:
        obj = world.objects[world.holding]
        hold_x, hold_y = 8, 3.5
        
        if obj.shape == "pyramid":
            points = [
                [hold_x, hold_y],
                [hold_x + 0.5, hold_y],
                [hold_x + 0.25, hold_y + 0.6]
            ]
            triangle = patches.Polygon(points, closed=True,
                                     edgecolor='black', facecolor=obj.color,
                                     linewidth=2, linestyle='--')
            ax.add_patch(triangle)
        else:
            rect = patches.Rectangle((hold_x, hold_y), 0.5, 0.5,
                                    edgecolor='black', facecolor=obj.color,
                                    linewidth=2, linestyle='--')
            ax.add_patch(rect)
        
        ax.text(hold_x + 0.25, hold_y - 0.3, f"HOLDING: {world.holding}",
               ha='center', fontsize=10, fontweight='bold')
    
    # Draw table line
    ax.axhline(y=0, color='brown', linewidth=4, linestyle='-', alpha=0.7)
    ax.text(5, -0.3, "TABLE", ha='center', fontsize=12, fontweight='bold', color='brown')
    
    plt.tight_layout()
    
    # Convert to image
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf)
    plt.close(fig)
    
    return img

In [15]:
# ----------------------------------------
# STEP 1) Processing
# ----------------------------------------

def process_command(command: str, history: List[List[str]]) -> Tuple[List[List[str]], Image.Image, str]:
    """Process a command and update the world."""
    global world_state
    
    if not command.strip():
        return history, visualize_world(world_state), world_state.describe()
    
    try:
        response = interpret_and_act(world_state, command)
        history.append([command, response])
    except (ValueError, RuntimeError) as e:
        response = f"❌ {str(e)}"
        history.append([command, response])
    
    return history, visualize_world(world_state), world_state.describe()

def reset_world():
    """Reset the world to initial state."""
    global world_state
    world_state = create_initial_world()
    return [], visualize_world(world_state), world_state.describe()

# Build Gradio interface
with gr.Blocks(title="SHRDLU Blocks World") as demo:
    gr.Markdown("""
    # 🧱 SHRDLU Blocks World Simulator
    
    A modern implementation of Terry Winograd's classic SHRDLU natural language understanding system.
    
    **Try commands like:**
    - "pick up the blue pyramid"
    - "put the blue pyramid on the red cube"
    - "put the green cube on the blue pyramid"
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="Conversation History",
                height=400,
                show_label=True,
                # bubble_full_width=False
            )
            
            with gr.Row():
                command_input = gr.Textbox(
                    label="Enter Command",
                    placeholder="e.g., put the blue pyramid on the red cube",
                    scale=4
                )
                submit_btn = gr.Button("Execute", variant="primary", scale=1)
            
            with gr.Row():
                reset_btn = gr.Button("🔄 Reset World", variant="secondary")
            
            world_text = gr.Textbox(
                label="World State (Text)",
                lines=6,
                interactive=False
            )
        
        with gr.Column(scale=1):
            world_viz = gr.Image(
                label="World Visualization",
                type="pil",
                height=500
            )
    
    gr.Markdown("""
    ### 📝 Notes:
    - Objects are identified by **color** and **shape**
    - The system will ask for clarification if the description is ambiguous
    - The planner automatically moves blocking objects out of the way
    - Dashed outline indicates an object being held
    """)
    
    # Event handlers
    submit_btn.click(
        fn=process_command,
        inputs=[command_input, chatbot],
        outputs=[chatbot, world_viz, world_text]
    ).then(
        lambda: "",
        outputs=[command_input]
    )
    
    command_input.submit(
        fn=process_command,
        inputs=[command_input, chatbot],
        outputs=[chatbot, world_viz, world_text]
    ).then(
        lambda: "",
        outputs=[command_input]
    )
    
    reset_btn.click(
        fn=reset_world,
        outputs=[chatbot, world_viz, world_text]
    )
    
    # Load initial state
    demo.load(
        fn=lambda: ([], visualize_world(world_state), world_state.describe()),
        outputs=[chatbot, world_viz, world_text]
    )

if __name__ == "__main__":
    demo.launch(share=False, server_name="0.0.0.0")


C:\Users\kyria\AppData\Local\Temp\ipykernel_21612\3011042430.py:42: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7860): only one usage of each socket address (protocol/network address/port) is normally permitted


* Running on local URL:  http://0.0.0.0:7861
* To create a public link, set `share=True` in `launch()`.


p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 None
b3 None
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 None
b3 None
c3 None
c4 None
p1 c1
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 None
b3 None
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 None
b3 None
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 None
b3 None
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 None
b3 None
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 None
b3 None
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 None
b3 None
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 None
b3 None
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 Non